In [3]:
import os
import librosa
import matplotlib.pyplot as plt
%matplotlib inline 
import numpy as np
import IPython.display as ipd 
from scipy.io import wavfile

In [12]:
train_dir = "train/"

samples,sample_rate = librosa.load(path=train_dir+"cat/0b09edd3_nohash_0.wav", sr=16000)

ipd.Audio(samples,rate=sample_rate)

In [9]:
samples = librosa.resample(samples,orig_sr=sample_rate,target_sr=8000)
ipd.Audio(samples,rate=8000)

In [13]:
labels = os.listdir(train_dir)
labels

['bed', 'bird', 'cat', 'dog', 'down', 'eight', 'five', 'four']

In [ ]:
no_of_recordings = []

for label in labels:
    waves = [f for f in os.listdir(train_dir + '/' + label) if f.endswith(".wav")]
    no_of_recordings.append(len(waves))

print(no_of_recordings)

[1713, 1731, 1733, 1746, 2359, 2352, 2357, 632]


In [15]:
labels=["bed", "bird", "cat", "dog", "down", "eight", "five", "four"]

In [16]:
train_dir = "train/"

all_wave = []
all_label = []

for label in labels:
    
    waves = [f for f in os.listdir(train_dir + "/" + label) if f.endswith('.wav')]
    
    for wave in waves:
        
        samples,sample_rate = librosa.load(path=train_dir + "/" + label + "/" + wave,sr=16000)
        
        samples = librosa.resample(samples, orig_sr=sample_rate, target_sr=8000)
        if(len(samples)== 8000) : 
            all_wave.append(samples)
            all_label.append(label)

In [17]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y=le.fit_transform(all_label)

In [28]:
classes= list(le.classes_)
print(classes)

['bed', 'bird', 'cat', 'dog', 'down', 'eight', 'five', 'four']


In [18]:
from keras.utils import np_utils
y=np_utils.to_categorical(y, num_classes=len(labels))
print(y)

[[1. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 1.]]


In [19]:
all_wave = np.array(all_wave).reshape(-1,8000,1)

In [20]:

from sklearn.model_selection import train_test_split
x_tr, x_val, y_tr, y_val = train_test_split(np.array(all_wave),np.array(y),stratify=y,test_size = 0.2,random_state=777,shuffle=True)

In [21]:
from keras.layers import Dense, Dropout, Flatten, Conv1D, Input, MaxPooling1D
from keras.models import Model
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras import backend as K
K.clear_session()

inputs = Input(shape=(8000,1))

#First Conv1D layer
conv = Conv1D(8,13, padding='valid', activation='relu', strides=1)(inputs)
conv = MaxPooling1D(3)(conv)
conv = Dropout(0.3)(conv)

#Second Conv1D layer
conv = Conv1D(16, 11, padding='valid', activation='relu', strides=1)(conv)
conv = MaxPooling1D(3)(conv)
conv = Dropout(0.3)(conv)

#Third Conv1D layer
conv = Conv1D(32, 9, padding='valid', activation='relu', strides=1)(conv)
conv = MaxPooling1D(3)(conv)
conv = Dropout(0.3)(conv)

#Fourth Conv1D layer
conv = Conv1D(64, 7, padding='valid', activation='relu', strides=1)(conv)
conv = MaxPooling1D(3)(conv)
conv = Dropout(0.3)(conv)

#Flatten layer
conv = Flatten()(conv)

#Dense Layer 1
conv = Dense(256, activation='relu')(conv)
conv = Dropout(0.3)(conv)

#Dense Layer 2
conv = Dense(128, activation='relu')(conv)
conv = Dropout(0.3)(conv)

outputs = Dense(len(labels), activation='softmax')(conv)

model = Model(inputs, outputs)
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 8000, 1)]         0         
                                                                 
 conv1d (Conv1D)             (None, 7988, 8)           112       
                                                                 
 max_pooling1d (MaxPooling1D  (None, 2662, 8)          0         
 )                                                               
                                                                 
 dropout (Dropout)           (None, 2662, 8)           0         
                                                                 
 conv1d_1 (Conv1D)           (None, 2652, 16)          1424      
                                                                 
 max_pooling1d_1 (MaxPooling  (None, 884, 16)          0         
 1D)                                                         

In [22]:
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [23]:
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10, min_delta=0.0001) 
mc = ModelCheckpoint('voiceRecModel.hdf5', monitor='val_acc', verbose=1, save_best_only=True, mode='max')
print(x_tr.shape, x_val.shape, y_tr.shape, y_val.shape)

(10448, 8000, 1) (2613, 8000, 1) (10448, 8) (2613, 8)


In [24]:
history=model.fit(x_tr, y_tr ,epochs=100, callbacks=[es,mc], batch_size=32, validation_data=(x_val,y_val))

Epoch 1/100
327/327 [==============================] - 32s 95ms/step - loss: 1.8096 - accuracy: 0.3008 - val_loss: 1.6145 - val_accuracy: 0.3911
Epoch 2/100
327/327 [==============================] - 30s 90ms/step - loss: 1.4863 - accuracy: 0.4501 - val_loss: 1.2412 - val_accuracy: 0.5438
Epoch 3/100
327/327 [==============================] - 32s 98ms/step - loss: 1.1196 - accuracy: 0.5968 - val_loss: 0.9108 - val_accuracy: 0.6931
Epoch 4/100
327/327 [==============================] - 31s 96ms/step - loss: 0.9217 - accuracy: 0.6755 - val_loss: 0.7226 - val_accuracy: 0.7627
Epoch 5/100
327/327 [==============================] - 32s 98ms/step - loss: 0.7953 - accuracy: 0.7229 - val_loss: 0.8217 - val_accuracy: 0.7264
Epoch 6/100
327/327 [==============================] - 32s 96ms/step - loss: 0.6797 - accuracy: 0.7718 - val_loss: 0.5850 - val_accuracy: 0.8117
Epoch 7/100
327/327 [==============================] - 31s 96ms/step - loss: 0.5840 - accuracy: 0.8040 - val_loss: 0.5441 - val_ac

KeyboardInterrupt: 

In [ ]:
model.save("voiceRecModel.hdf5")

In [39]:
from keras.models import load_model
# from PIL import Image

model = load_model("voiceRecModel.hdf5")

In [40]:
def find(audio):
    prob=model.predict(audio.reshape(1,8000,1))
    index=np.argmax(prob[0])
    return classes[index]

import random
index=random.randint(0,len(x_val)-1)
samples=x_val[index].ravel()
print("Audio:",classes[np.argmax(y_val[index])])
ipd.Audio(samples, rate=8000)

Audio: down


In [41]:
print("Text:",find(samples))

Text: down
